In [ ]:
!pip install nltk
import nltk
nltk.download('punkt_tab')
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from collections import defaultdict
import nltk # Ensure nltk is imported if not already

# --- Ensure NLTK data is downloaded (Add these checks if not present) ---
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
# --- End NLTK Downloads ---


# --- Preprocessing Function (Keep as is) ---
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    """Cleans and preprocesses text data."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    # Keep lemmatization if used in training, otherwise remove lemmatizer parts
    # lemmatizer = WordNetLemmatizer() # Uncomment if you lemmatized during training
    processed_tokens = [
        # lemmatizer.lemmatize(word) # Uncomment if lemmatizing
        word # Comment out if lemmatizing
        for word in tokens if word not in stop_words and word.isalpha()
    ]
    return ' '.join(processed_tokens)

# --- Prediction Function (Corrected) ---
def predict_sentiment(review_text, model, tokenizer, aspects, int_to_str_map):
    """Predicts sentiment for all aspects for a given review text using the fine-tuned model."""
    processed_text = preprocess_text(review_text)
    if not processed_text:
        return {"error": "Review text is empty after preprocessing."}

    inputs = tokenizer(processed_text, return_tensors='pt', truncation=True, padding=True, max_length=128) # Adjust max_length if needed

    # Ensure inputs are on the same device as the model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits # Shape: (batch_size, num_aspects * num_classes_per_aspect)

    # --- Process Logits for Multi-Output Classification ---
    num_aspects = len(aspects)
    num_classes = len(int_to_str_map)

    # Reshape logits to (batch_size, num_aspects, num_classes)
    # Ensure the model output size matches num_aspects * num_classes
    expected_logit_size = num_aspects * num_classes
    if logits.shape[1] != expected_logit_size:
         return {"error": f"Model output size ({logits.shape[1]}) does not match expected size ({expected_logit_size} = {num_aspects} aspects * {num_classes} classes). Was the model trained correctly for multi-output?"}

    reshaped_logits = logits.view(1, num_aspects, num_classes) # Batch size is 1 here

    # Get the predicted class index for each aspect by finding the max logit along the class dimension (dim=2)
    prediction_indices = torch.argmax(reshaped_logits, dim=2).squeeze().cpu().numpy() # Shape: (num_aspects,)

    # Map indices to sentiment labels
    predicted_sentiments = {}
    if prediction_indices.shape == (): # Handle case where there's only one aspect (scalar output)
        prediction_indices = [prediction_indices.item()] # Convert scalar to list

    if len(prediction_indices) != num_aspects:
         return {"error": f"Number of predictions ({len(prediction_indices)}) does not match number of aspects ({num_aspects})."}


    for i, aspect_name in enumerate(aspects):
        pred_idx = prediction_indices[i]
        predicted_sentiments[aspect_name] = int_to_str_map.get(pred_idx, "Unknown Label") # Use .get() for safety

    return predicted_sentiments

# --- Main Execution Block (Modified) ---

# Load your fine-tuned model and tokenizer
# Ensure these paths are correct for your environment
save_directory = '/content/drive/My Drive/Colab Notebooks' # Or your specific path

try:
    # Mount Google Drive if in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    model = AutoModelForSequenceClassification.from_pretrained(save_directory)
    tokenizer = AutoTokenizer.from_pretrained(save_directory)
    print("Model and tokenizer loaded successfully from Google Drive.")
    # Move model to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print(f"Using device: {device}")

except ModuleNotFoundError:
    print("Not running in Google Colab or drive not mounted. Loading locally.")
    # Adjust local path if necessary
    # save_directory = 'path/to/your/local/model'
    try:
        model = AutoModelForSequenceClassification.from_pretrained(save_directory)
        tokenizer = AutoTokenizer.from_pretrained(save_directory)
        print("Model and tokenizer loaded successfully locally.")
        # Move model to GPU if available
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)
        print(f"Using device: {device}")
    except Exception as e:
        print(f"Error loading model or tokenizer locally from {save_directory}: {e}")
        print("Please ensure the path is correct and the model files exist.")
        exit()
except Exception as e:
    print(f"Error loading model or tokenizer from Google Drive {save_directory}: {e}")
    print("Please ensure the path is correct, Drive is mounted, and model files exist.")
    exit()


# Aspects the model was trained on (ensure this matches training)
aspects = ['food', 'service', 'ambiance', 'price', 'context', 'overall'] # [cite: 92]

# Mapping of integer labels (used during training) to sentiment strings
# Verify this matches the labels used when training (e.g., 0, 1, 2) [cite: 86]
integer_to_string_label_map = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}
# Note: The original PDF example used {-1: 'negative', 0: 'neutral', 1: 'positive'} for prediction output[cite: 128],
# but training likely used {0, 1, 2} based on the binning code[cite: 86].
# Adjust the map above if your training labels were different.


# --- Interactive Sentiment Prediction ---
while True:
    user_review = input("Enter a restaurant review (or type 'exit' to quit): ")
    if user_review.lower() == 'exit':
        break

    # Use the corrected prediction function
    predicted_sentiments = predict_sentiment(user_review, model, tokenizer, aspects, integer_to_string_label_map)

    print("\n--- Predicted Sentiments ---")
    if isinstance(predicted_sentiments, dict) and "error" in predicted_sentiments:
         print(f"Error: {predicted_sentiments['error']}")
    elif isinstance(predicted_sentiments, dict):
        for aspect, sentiment in predicted_sentiments.items():
             # Add .capitalize() if the labels in the map are lowercase
            print(f"- {aspect.capitalize()}: {sentiment.capitalize()}")
    else:
        # Handle unexpected output format from predict_sentiment if necessary
        print("An unexpected error occurred during prediction.")


    print("-" * 30)

print("\nExiting sentiment prediction.")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Mounted at /content/drive
Model and tokenizer loaded successfully from Google Drive.
Using device: cuda

--- Predicted Sentiments ---
- Food: Neutral
- Service: Neutral
- Ambiance: Neutral
- Price: Neutral
- Context: Neutral
- Overall: Positive
------------------------------
